# McDonald's diet problem

Determine the minimum-cost McDonald's menu that meets all of the nutrient minimums in the supplied classroom data. We will solve the same linear program twice: first with named indices and a `NamedArray`, then with integer indices and ordinary arrays.

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. JuMP, HiGHS, NamedArrays, and Printf are already included in that environment. No external data files are needed.

This LP allows fractional servings and uses the prices and nutrient values from the original classroom example.

## The mathematical model

Let $F$ be the set of foods and $N$ the set of nutrients. Let $c_j$ be the cost of one serving of food $j$, $a_{ij}$ its amount of nutrient $i$, and $b_i$ the required minimum of nutrient $i$. The decision variable $x_j$ is the number of servings of food $j$.

$$
\begin{aligned}
\min_x\quad & \sum_{j\in F} c_j x_j \\
\text{subject to}\quad & \sum_{j\in F} a_{ij}x_j \geq b_i && \forall i\in N, \\
& x_j \geq 0 && \forall j\in F.
\end{aligned}
$$

Every nutrient constraint is a lower bound. The model has no upper limits on nutrient intake and no whole-serving restrictions.

## 1. Named indices

The food symbols stand for:

| Symbol | Menu item |
| --- | --- |
| `:QP` | Quarter Pounder |
| `:MD` | McLean Deluxe |
| `:BM` | Big Mac |
| `:FF` | Filet-O-Fish |
| `:MC` | McGrilled Chicken |
| `:FR` | Small Fries |
| `:SM` | Sausage McMuffin |
| `:M1` | 1% Milk |
| `:OJ` | Orange Juice |

The nutrient rows are protein, vitamin A, vitamin C, calcium, iron, calories, and carbohydrates, in that order. Each minimum uses the same units as its corresponding row of `A`.

`Dict(zip(labels, values))` pairs each label with its value. `NamedArray` adds row and column labels to a numerical array; `A_NA[:Prot, :QP]` is the protein in one Quarter Pounder. We use `:M1` because an identifier cannot begin with a digit.

In [ ]:
using JuMP, HiGHS, NamedArrays, Printf
import MathOptInterface as MOI

foods = [:QP, :MD, :BM, :FF, :MC, :FR, :SM, :M1, :OJ]
nutrients = [:Prot, :VitA, :VitC, :Calc, :Iron, :Cals, :Carb]

cost = Dict(zip(foods, [1.84, 2.19, 1.84, 1.44, 2.29, 0.77, 1.29, 0.6, 0.72]))
required = Dict(zip(nutrients, [55, 100, 100, 100, 100, 2000, 350]))

# Rows are nutrients; columns are foods, in the orders given above.
A = [
    28 24 25 14 31 3 15 9 1
    15 15 6 2 8 0 4 10 2
    6 10 2 0 15 15 0 4 120
    30 20 25 15 15 0 20 30 2
    20 20 20 10 8 2 15 0 2
    510 370 500 370 400 220 345 110 80
    34 33 42 38 42 26 27 12 20
]
A_NA = NamedArray(A, (nutrients, foods), ("Nutrients", "Menu Items"))
A_NA

In [ ]:
named_model = Model(HiGHS.Optimizer)
set_silent(named_model)

@variable(named_model, x_named[foods] >= 0)
@objective(named_model, Min, sum(cost[j] * x_named[j] for j in foods))
@constraint(named_model, nutrient_minimum[i in nutrients],
    sum(A_NA[i, j] * x_named[j] for j in foods) >= required[i])

named_model

### Solve, check, and report

After `optimize!`, check that HiGHS found an optimum and that a feasible solution is available before reading variable or objective values.

The conditional dictionary comprehension keeps only foods with more than `1e-6` servings, to omit numerical values close to zero. `@printf` formats the report: `%.2f` prints two decimal places and `%s` prints a label. Iterate over `foods` to report the solution in menu order.

In [ ]:
optimize!(named_model)
named_status = termination_status(named_model)
named_status == MOI.OPTIMAL || error("HiGHS stopped with status $(named_status).")
is_solved_and_feasible(named_model) || error("No feasible optimal solution is available.")

named_cost = objective_value(named_model)
named_solution = Dict(j => value(x_named[j]) for j in foods if value(x_named[j]) > 1e-6)

println("Termination status: ", named_status)
@printf("\nMinimum cost menu is \$%.2f\n", named_cost)
for j in foods
    if haskey(named_solution, j)
        @printf("Eat %.2f servings of menu item %s\n", named_solution[j], j)
    end
end

## 2. Integer indices

We can express the same model with integer indices. `A[i, j]` is still the amount of nutrient `i` in food `j`. The ordinary vectors below put costs and requirements in the same order as the columns and rows of `A`.

We reuse the numerical matrix and build a fresh model. Separate variable names keep both solutions available for comparison. Here, **integer indices** refer to the positions in arrays; the decision variables still allow fractional servings.

In [ ]:
food_names = [
    "Quarter Pounder", "McLean Deluxe", "Big Mac", "Filet-O-Fish",
    "McGrilled Chicken", "Small Fries", "Sausage McMuffin", "1% Milk", "Orange Juice",
]
cost_vector = [cost[j] for j in foods]
required_vector = [required[i] for i in nutrients]

M, N = size(A)  # 7 nutrient rows and 9 food columns

In [ ]:
indexed_model = Model(HiGHS.Optimizer)
set_silent(indexed_model)

@variable(indexed_model, x_indexed[1:N] >= 0)
@objective(indexed_model, Min, sum(cost_vector[j] * x_indexed[j] for j in 1:N))
@constraint(indexed_model, nutrient_minimum[i in 1:M],
    sum(A[i, j] * x_indexed[j] for j in 1:N) >= required_vector[i])

indexed_model

In [ ]:
optimize!(indexed_model)
indexed_status = termination_status(indexed_model)
indexed_status == MOI.OPTIMAL || error("HiGHS stopped with status $(indexed_status).")
is_solved_and_feasible(indexed_model) || error("No feasible optimal solution is available.")

indexed_cost = objective_value(indexed_model)
indexed_solution = Dict(
    food_names[j] => value(x_indexed[j]) for j in 1:N if value(x_indexed[j]) > 1e-6
)

println("Termination status: ", indexed_status)
@printf("\nMinimum cost menu is \$%.2f\n", indexed_cost)
for food in food_names
    if haskey(indexed_solution, food)
        @printf("Eat %.2f servings of %s\n", indexed_solution[food], food)
    end
end

@assert isapprox(named_cost, indexed_cost; atol = 1e-8)
println("\nBoth formulations have the same optimal cost.")

## Interpret the solution

Compare the food symbols in the first report with the full names in the second. The two formulations use the same data and constraints, so they describe the same LP.

The displayed servings are rounded to two decimal places; calculations use the unrounded values. Rounding a solution can violate a nutrient minimum. Requiring whole servings would change this LP into an integer program.

Which nutrient minimums are met exactly, and which are exceeded? The next cell computes the totals using the unrounded solution.

In [ ]:
nutrient_totals = Dict(
    i => sum(A_NA[i, j] * value(x_named[j]) for j in foods) for i in nutrients
)
for i in nutrients
    @printf("%s: total = %.2f, minimum = %.2f, surplus = %.2f\n",
        i, nutrient_totals[i], required[i], nutrient_totals[i] - required[i])
end